# Qwen2.5-0.5B vLLM Text Generation Demo

Three protocols: REST Generate, gRPC Streaming, REST Generate Stream

**Note:** The vLLM backend uses a decoupled (streaming) transaction policy, so the standard
Triton `/infer` HTTP endpoint is not supported. Use `/generate` for HTTP and streaming infer for gRPC.

In [ ]:
import os
import json
import time
import numpy as np
import requests
import tritonclient.grpc as grpcclient

# =============================================================================
# Configuration - Update NAMESPACE for your environment
# =============================================================================
NAMESPACE = "domino-inference-dev"
#NAMESPACE = "domino-inference-test"
#NAMESPACE = "domino-inference-prod"

os.environ["TRITON_GRPC_URL"] = f"triton-inference-server-proxy.{NAMESPACE}.svc.cluster.local:50051"
os.environ["TRITON_REST_URL"] = f"http://triton-inference-server-proxy.{NAMESPACE}.svc.cluster.local:8080"

GRPC_URL = os.environ.get("TRITON_GRPC_URL", "localhost:50051")
REST_URL = os.environ.get("TRITON_REST_URL", "http://localhost:8080")

API_KEY = os.environ.get("DOMINO_USER_API_KEY", "")
grpc_headers = {"x-domino-api-key": API_KEY} if API_KEY else None
http_headers = {"X-Domino-Api-Key": API_KEY} if API_KEY else {}

print(f"Namespace: {NAMESPACE}")
print(f"gRPC URL: {GRPC_URL}")
print(f"REST URL: {REST_URL}")
print(f"Auth: {'API Key configured' if API_KEY else 'Disabled'}")

In [ ]:
# =============================================================================
# Prepare inputs
# =============================================================================
prompt = "What is the capital of France?"
max_tokens = 128
temperature = 0.7
top_p = 0.9

print(f"Prompt: {prompt}")
print(f"Max tokens: {max_tokens}")

In [ ]:
# =============================================================================
# 1. REST - /generate endpoint (non-streaming)
# vLLM decoupled models do not support the standard /infer HTTP endpoint.
# Use /v2/models/{model}/generate instead.
# =============================================================================
payload = {
    "text_input": prompt,
    "parameters": {
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
        "stream": False,
    }
}

start = time.time()
resp = requests.post(
    f"{REST_URL}/v2/models/qwen25-vllm/generate",
    json=payload,
    headers=http_headers,
)
resp.raise_for_status()
t_rest = (time.time() - start) * 1000
text_rest = resp.json()["text_output"]

print(f"✓ REST (generate): {t_rest:8.2f} ms")
print(f"Response: {text_rest}")

In [ ]:
# =============================================================================
# 2. gRPC - async streaming infer (required for decoupled models)
# Uses tritonclient.grpc.aio which works natively in Jupyter's event loop.
# stream_infer() takes an async generator that yields dicts with model_name + inputs.
# =============================================================================
import tritonclient.grpc.aio as aio_grpcclient

async def infer_grpc():
    client = aio_grpcclient.InferenceServerClient(url=GRPC_URL)

    inp_text = aio_grpcclient.InferInput("text_input", [1], "BYTES")
    inp_text.set_data_from_numpy(np.array([prompt], dtype=np.object_))
    inp_params = aio_grpcclient.InferInput("sampling_parameters", [1], "BYTES")
    inp_params.set_data_from_numpy(np.array(
        [json.dumps({"temperature": temperature, "top_p": top_p, "max_tokens": max_tokens})],
        dtype=np.object_
    ))
    inp_stream = aio_grpcclient.InferInput("stream", [1], "BOOL")
    inp_stream.set_data_from_numpy(np.array([False], dtype=bool))

    outputs = [aio_grpcclient.InferRequestedOutput("text_output")]

    async def request_iterator():
        yield {
            "model_name": "qwen25-vllm",
            "inputs": [inp_text, inp_params, inp_stream],
            "outputs": outputs,
        }

    result_text = ""
    async for result, error in client.stream_infer(
        request_iterator(),
        headers=grpc_headers,
    ):
        if error:
            raise error
        token = result.as_numpy("text_output")[0]
        if isinstance(token, bytes):
            token = token.decode("utf-8")
        result_text += token

    await client.close()
    return result_text

start = time.time()
text_grpc = await infer_grpc()
t_grpc = (time.time() - start) * 1000

print(f"✓ gRPC (streaming): {t_grpc:8.2f} ms")
print(f"Response: {text_grpc}")

In [ ]:
# =============================================================================
# 3. REST - /generate_stream endpoint (streaming)
# =============================================================================
payload_stream = {
    "text_input": prompt,
    "parameters": {
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
        "stream": True,
    }
}

start = time.time()
text_stream = ""
with requests.post(
    f"{REST_URL}/v2/models/qwen25-vllm/generate_stream",
    json=payload_stream,
    headers=http_headers,
    stream=True,
) as resp:
    resp.raise_for_status()
    for line in resp.iter_lines():
        if line:
            # SSE format: "data: {...}"
            data = line.decode("utf-8").removeprefix("data: ")
            chunk = json.loads(data)
            text_stream += chunk.get("text_output", "")
t_stream = (time.time() - start) * 1000

print(f"✓ REST (generate_stream): {t_stream:8.2f} ms")
print(f"Response: {text_stream}")

In [ ]:
# =============================================================================
# Results
# =============================================================================
print("Generated text (gRPC):")
print(text_grpc)